# Module 5.02: Fine-tuning with LoRA (Low-Rank Adaptation)


In [ ]:
# Install dependencies (run only once)
# Note: Installing specific versions to avoid compatibility issues
!pip install transformers datasets peft accelerate torch
!pip install matplotlib seaborn pandas numpy

# Optional: Only install bitsandbytes if you want 8-bit training (not needed for basic LoRA)
# Uncomment the next line only if you have GPU and want quantized training
# !pip install bitsandbytes

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import os
os.environ["PEFT_DISABLE_TORCHAO"] = "1"

!pip uninstall torchao -y -q
!pip install transformers==4.44.0 peft==0.12.0 accelerate==0.33.0 datasets==2.21.0 -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 49.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>

## 1. Dataset Preparation and Specific Task

For this example, we'll train a small model to write **scientific article summaries** in a specific style.

In [ ]:
# Example dataset: scientific articles and their summaries
training_data = [
    {
        "instruction": "Write a summary of the following scientific article in maximum 50 words:",
        "input": "Deep learning models have shown remarkable performance in natural language processing tasks. However, they require substantial computational resources and large datasets for training. Recent advances in transfer learning have made it possible to adapt pre-trained models to specific tasks with limited data, reducing both training time and computational requirements.",
        "output": "Deep learning models excel in NLP but require extensive resources. Transfer learning enables adapting pre-trained models to specific tasks with limited data, reducing training time and computational costs significantly."
    },
    {
        "instruction": "Write a summary of the following scientific article in maximum 50 words:",
        "input": "Attention mechanisms have revolutionized neural machine translation by allowing models to focus on relevant parts of the input sequence. The Transformer architecture, based entirely on attention, has achieved state-of-the-art results across multiple NLP tasks while being more parallelizable than recurrent approaches.",
        "output": "Attention mechanisms revolutionized neural machine translation by enabling models to focus on relevant input parts. Transformers, based solely on attention, achieve state-of-the-art NLP results with greater parallelization than recurrent approaches."
    },
    {
        "instruction": "Write a summary of the following scientific article in maximum 50 words:",
        "input": "Computer vision applications in medical imaging have shown promising results for diagnosis assistance. Convolutional neural networks can detect anomalies in X-rays, CT scans, and MRI images with accuracy comparable to expert radiologists. However, interpretability and bias issues remain significant challenges for clinical deployment.",
        "output": "Computer vision in medical imaging shows promising diagnostic assistance results. CNNs detect anomalies in X-rays, CT scans, and MRIs with radiologist-level accuracy, but interpretability and bias issues remain challenges for clinical deployment."
    },
    {
        "instruction": "Write a summary of the following scientific article in maximum 50 words:",
        "input": "Reinforcement learning algorithms have achieved superhuman performance in complex games like Go and Chess. These successes demonstrate the potential of RL for strategic decision making. Current research focuses on applying similar techniques to real-world problems like robotics, autonomous driving, and resource optimization.",
        "output": "Reinforcement learning algorithms achieve superhuman performance in complex games like Go and Chess. These successes demonstrate RL's potential for strategic decision-making. Current research focuses on real-world applications like robotics and autonomous driving."
    },
    {
        "instruction": "Write a summary of the following scientific article in maximum 50 words:",
        "input": "Natural language generation has improved dramatically with large language models. These models can produce coherent text across various domains and styles. However, issues with factual accuracy and tendency to hallucinate information remain concerns for practical applications requiring reliability.",
        "output": "Natural language generation improved dramatically with large language models, producing coherent text across various domains and styles. However, factual accuracy issues and hallucination tendencies remain concerns for reliable practical applications."
    }
]

# Convert to training format
def format_instruction(sample):
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

# Create formatted dataset
formatted_data = [format_instruction(item) for item in training_data]

print("Example formatted data:")
print(formatted_data[0])
print(f"\nTotal number of examples: {len(formatted_data)}")

Example formatted data:
### Instruction:
Write a summary of the following scientific article in maximum 50 words:

### Input:
Deep learning models have shown remarkable performance in natural language processing tasks. However, they require substantial computational resources and large datasets for training. Recent advances in transfer learning have made it possible to adapt pre-trained models to specific tasks with limited data, reducing both training time and computational requirements.

### Response:
Deep learning models excel in NLP but require extensive resources. Transfer learning enables adapting pre-trained models to specific tasks with limited data, reducing training time and computational costs significantly.

Total number of examples: 5


## 2. Base Model Loading

We'll use **GPT-2 small** (124M parameters) as the base model - perfect for CPU and educational purposes.

In [ ]:
# Load model and tokenizer
model_name = "gpt2"  # GPT-2 small (124M params)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 doesn't have pad_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # For CPU
    device_map="cpu"
)

print(f"Model loaded: {model_name}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Vocabulary size: {len(tokenizer)}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: gpt2
Number of parameters: 124,439,808
Vocabulary size: 50257


## 3. Base Model Testing (Before Fine-tuning)

Let's test the model's capabilities **before** training on an example from our task.

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=200, temperature=0.7, top_p=0.9):
    """Helper function for text generation"""
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test prompt
test_prompt = """### Instruction:
Write a summary of the following scientific article in maximum 50 words:

### Input:
Graph neural networks have emerged as powerful tools for processing structured data. Unlike traditional neural networks that work on Euclidean data, GNNs can handle irregular graph structures. They have shown excellent performance in social network analysis, molecular property prediction, and recommendation systems.

### Response:"""

print("=== BASE MODEL (before fine-tuning) ===")
base_output = generate_text(model, tokenizer, test_prompt)
print(base_output)
print("\n" + "="*60)

=== BASE MODEL (before fine-tuning) ===
### Instruction:
Write a summary of the following scientific article in maximum 50 words:

### Input:
Graph neural networks have emerged as powerful tools for processing structured data. Unlike traditional neural networks that work on Euclidean data, GNNs can handle irregular graph structures. They have shown excellent performance in social network analysis, molecular property prediction, and recommendation systems.

### Response:

GNNs can be used for both classification and prediction. GNNs can be used for both classification and prediction.

### Abstract:

The main purpose of this paper is to provide a list of the most important aspects of the human brain and its processing. It is intended to provide a summary of the most important aspects of the human brain and its processing. It is also intended to provide a comprehensive review of the literature on GNNs.

### References:

[1] S.S. Dibb, S.M. Huygens



In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,  # Rank of LoRA matrices (lower = more efficient)
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.1,  # Dropout for regularization
    target_modules=["c_attn", "c_proj"],  # Modules to adapt in GPT-2
    bias="none",
    inference_mode=False,
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Parameter statistics
model.print_trainable_parameters()

print("\n=== LoRA Configuration ===")
print(f"Rank: {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Target modules: {lora_config.target_modules}")
print(f"Dropout: {lora_config.lora_dropout}")

trainable params: 811,008 || all params: 125,250,816 || trainable%: 0.6475

=== LoRA Configuration ===
Rank: 8
Alpha: 32
Target modules: {'c_proj', 'c_attn'}
Dropout: 0.1


## 5. Training Dataset Preparation

In [ ]:
# Dataset tokenization
def tokenize_function(examples):
    # Tokenize texts
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=512,
        return_overflowing_tokens=False,
    )

    # For causal LM, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Create Hugging Face dataset
train_dataset = Dataset.from_dict({"text": formatted_data})
train_dataset = train_dataset.map(tokenize_function, batched=True)

print(f"Training dataset: {len(train_dataset)} examples")
print(f"Average token length: {np.mean([len(x) for x in train_dataset['input_ids']]):.1f}")

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
    return_tensors="pt"
)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Training dataset: 5 examples
Average token length: 121.4


## 6. LoRA Training

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="./lora-gpt2-summaries",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Small for CPU
    gradient_accumulation_steps=2,  # Simulates batch_size=4
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    prediction_loss_only=True,
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to=[],  # Disable wandb
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Training configuration completed!")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training configuration completed!
Epochs: 3
Batch size: 2
Learning rate: 5e-05


In [ ]:
# Debug the tokenization
print("Checking tokenization...")
sample_data = train_dataset[0]
print("Sample data keys:", sample_data.keys())
print("Input_ids type:", type(sample_data['input_ids']))
print("Input_ids shape:", len(sample_data['input_ids']) if hasattr(sample_data['input_ids'], '__len__') else 'No length')
print("Labels type:", type(sample_data['labels']))

Checking tokenization...
Sample data keys: dict_keys(['text', 'input_ids', 'attention_mask', 'labels'])
Input_ids type: <class 'list'>
Input_ids shape: 120
Labels type: <class 'list'>


In [ ]:
# Remove the 'text' column that's causing the issue
train_dataset = train_dataset.remove_columns(['text'])

print("✅ Removed text column")
print(f"Dataset columns: {train_dataset.column_names}")
print(f"Sample data keys: {train_dataset[0].keys()}")

# Verify the data is clean
sample = train_dataset[0]
print(f"Input_ids type: {type(sample['input_ids'])}")
print(f"Labels type: {type(sample['labels'])}")
print(f"Attention_mask type: {type(sample['attention_mask'])}")

✅ Removed text column
Dataset columns: ['input_ids', 'attention_mask', 'labels']
Sample data keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input_ids type: <class 'list'>
Labels type: <class 'list'>
Attention_mask type: <class 'list'>


In [ ]:
# Test the data collator directly
print("Testing data collator...")
sample_batch = [train_dataset[0], train_dataset[1]]
print("Sample batch types:")
for i, sample in enumerate(sample_batch):
    print(f"Sample {i}:")
    for key, value in sample.items():
        print(f"  {key}: {type(value)} - {len(value) if hasattr(value, '__len__') else value}")

# Try the data collator
try:
    collated = data_collator(sample_batch)
    print("✅ Data collator works!")
except Exception as e:
    print(f"❌ Data collator error: {e}")

Testing data collator...
Sample batch types:
Sample 0:
  input_ids: <class 'list'> - 120
  attention_mask: <class 'list'> - 120
  labels: <class 'list'> - 120
Sample 1:
  input_ids: <class 'list'> - 129
  attention_mask: <class 'list'> - 129
  labels: <class 'list'> - 129
❌ Data collator error: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`labels` in this case) have excessive nesting (inputs type `list` where type `int` is expected).


In [ ]:
# Replace with a simpler data collator
from transformers import DataCollatorWithPadding

# Simpler data collator that handles the conversion better
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt"
)

print("✅ Using simpler data collator")

✅ Using simpler data collator


In [ ]:
# Start training
print("Starting LoRA fine-tuning...")
print("This may take several minutes on CPU.")
print("="*50)

trainer.train()

print("\nTraining completed!")
print("Saving model...")

# Save fine-tuned model
trainer.save_model()
tokenizer.save_pretrained("./lora-gpt2-summaries")

print("Model saved to './lora-gpt2-summaries'")

Starting LoRA fine-tuning...
This may take several minutes on CPU.


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`text` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

## 7. Fine-tuned Model Testing

Now let's test the model **after** fine-tuning on the same example as before.

In [ ]:
# Test with the same prompt as before
print("=== FINE-TUNED MODEL (after LoRA) ===")
tuned_output = generate_text(model, tokenizer, test_prompt)
print(tuned_output)
print("\n" + "="*60)

# Test on training set example to check overfitting
train_example = """### Instruction:
Write a summary of the following scientific article in maximum 50 words:

### Input:
Deep learning models have shown remarkable performance in natural language processing tasks. However, they require substantial computational resources and large datasets for training.

### Response:"""

print("=== TEST ON TRAINING SET EXAMPLE ===")
train_output = generate_text(model, tokenizer, train_example)
print(train_output)
print("\n" + "="*60)

## 8. Quantitative Before/After Comparison

Let's evaluate generation quality using appropriate metrics.

In [ ]:
def calculate_perplexity(model, tokenizer, text):
    """Calculate text perplexity"""
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs.input_ids)
        loss = outputs.loss
        perplexity = torch.exp(loss)

    return perplexity.item()

def count_words(text):
    """Count words in text"""
    return len(text.split())

def extract_response(generated_text, prompt):
    """Extract only the generated response"""
    if "### Response:" in generated_text:
        return generated_text.split("### Response:")[-1].strip()
    return generated_text.replace(prompt, "").strip()

# Create different test cases
test_cases = [
    "Graph neural networks have emerged as powerful tools for processing structured data.",
    "Quantum computing promises exponential speedup for certain computational problems.",
    "Blockchain technology enables decentralized systems without trusted intermediaries."
]

results = []

for i, test_input in enumerate(test_cases):
    prompt = f"""### Instruction:
Write a summary of the following scientific article in maximum 50 words:

### Input:
{test_input}

### Response:"""

    # Generate with fine-tuned model
    output = generate_text(model, tokenizer, prompt, max_length=150)
    response = extract_response(output, prompt)

    # Calculate metrics
    word_count = count_words(response)
    perplexity = calculate_perplexity(model, tokenizer, response)

    results.append({
        'Test': f'Case {i+1}',
        'Input': test_input[:50] + '...',
        'Generated_Response': response[:100] + '...',
        'Word_Count': word_count,
        'Perplexity': perplexity,
        'Follows_Format': '### Response:' in output
    })

# Display results
df_results = pd.DataFrame(results)
print("=== QUANTITATIVE RESULTS ===")
print(df_results[['Test', 'Word_Count', 'Perplexity', 'Follows_Format']])
print("\n=== EXAMPLES OF GENERATED RESPONSES ===")
for i, row in df_results.iterrows():
    print(f"\n{row['Test']}:")
    print(f"Input: {row['Input']}")
    print(f"Response: {row['Generated_Response']}")
    print(f"Words: {row['Word_Count']}, Perplexity: {row['Perplexity']:.2f}")

## 9. Results Analysis

Let's visualize the fine-tuned model's performance.

In [ ]:
# Results visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Response length distribution
axes[0].bar(df_results['Test'], df_results['Word_Count'])
axes[0].axhline(y=50, color='r', linestyle='--', label='Target (50 words)')
axes[0].set_title('Generated Response Lengths')
axes[0].set_ylabel('Number of words')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

# 2. Perplexity
axes[1].bar(df_results['Test'], df_results['Perplexity'], color='orange')
axes[1].set_title('Response Perplexity')
axes[1].set_ylabel('Perplexity')
axes[1].tick_params(axis='x', rotation=45)

# 3. Format adherence
format_counts = df_results['Follows_Format'].value_counts()
axes[2].pie(format_counts.values, labels=format_counts.index, autopct='%1.1f%%')
axes[2].set_title('Format Adherence')

plt.tight_layout()
plt.show()

# Summary statistics
print("=== SUMMARY STATISTICS ===")
print(f"Average response length: {df_results['Word_Count'].mean():.1f} words")
print(f"Average perplexity: {df_results['Perplexity'].mean():.2f}")
print(f"% Responses following format: {df_results['Follows_Format'].mean()*100:.1f}%")
print(f"% Responses within 50 words: {(df_results['Word_Count'] <= 50).mean()*100:.1f}%")

## 10. Comparison with Base Model

Let's reload the base model for direct comparison.

In [ ]:
# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(
    "gpt2",
    torch_dtype=torch.float32,
    device_map="cpu"
)

# Comparative test
test_prompt = """### Instruction:
Write a summary of the following scientific article in maximum 50 words:

### Input:
Artificial intelligence systems are becoming increasingly sophisticated but also more opaque. The lack of interpretability in deep learning models poses challenges for their deployment in critical applications where understanding the decision-making process is essential for trust and accountability.

### Response:"""

print("=== DIRECT COMPARISON ===")
print("\n1. BASE MODEL:")
base_output = generate_text(base_model, tokenizer, test_prompt, max_length=200)
base_response = extract_response(base_output, test_prompt)
print(base_response)
print(f"Words: {count_words(base_response)}")

print("\n2. FINE-TUNED MODEL:")
tuned_output = generate_text(model, tokenizer, test_prompt, max_length=200)
tuned_response = extract_response(tuned_output, test_prompt)
print(tuned_response)
print(f"Words: {count_words(tuned_response)}")

print("\n=== QUALITATIVE ANALYSIS ===")
print("Observations:")
print("- Base model often generates unrelated text or doesn't follow format")
print("- Fine-tuned model better follows instructions and required style")
print("- LoRA enabled adaptation with few trainable parameters")
print("- Training on few examples still produced visible improvements")

## 11. Model Saving and Reuse

The LoRA model can be easily saved and reloaded.

In [ ]:
# Model was already saved during training
# Let's demonstrate how to reload it

from peft import PeftModel
import os

# Reload base model + LoRA adapter
def load_lora_model(base_model_name, lora_path):
    """
    Load base model + LoRA adapter
    """
    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

    # Load LoRA adapter
    model_with_lora = PeftModel.from_pretrained(base_model, lora_path)

    return model_with_lora

print("=== SAVED MODEL INFORMATION ===")
print(f"Location: ./lora-gpt2-summaries")
print(f"LoRA adapter size: ~{os.path.getsize('./lora-gpt2-summaries/adapter_model.bin')/1024/1024:.1f} MB")
print(f"Base model: {model_name} (~500 MB)")
print("\nAdvantages:")
print("- Only LoRA adapter is saved (~few MB)")
print("- Can be combined with any GPT-2 model")
print("- Easily distributable and versionable")

# Reloading example (commented to avoid confusion)
# reloaded_model = load_lora_model("gpt2", "./lora-gpt2-summaries")
# test_output = generate_text(reloaded_model, tokenizer, test_prompt)

## 12. Considerations and Best Practices

### LoRA Advantages:
- **Efficiency**: Only 0.1% of parameters to train
- **Speed**: Much faster training
- **Memory**: Requires much less RAM
- **Flexibility**: Interchangeable adapters for different tasks

### When to use LoRA:
- Fine-tuning large models with limited resources
- Adapting to specific tasks with few data
- Creating multiple specialized versions of the same base model
- Rapid prototyping of NLP applications

### Limitations:
- Potentially lower performance than full fine-tuning
- Works better on tasks similar to pre-training
- Still requires quality dataset

### Important parameters:
- **r (rank)**: 4-16 for simple tasks, 32-64 for complex tasks
- **alpha**: Often 2x the rank (r=8 → alpha=16)
- **target_modules**: Depends on model architecture
- **dropout**: 0.1 is a good default


## Practical Exercises

**Exercise 1**: Try changing the LoRA rank (r=4, r=16, r=32) and observe the effect on quality and training time.

**Exercise 2**: Create a dataset for a different task (e.g. English-Italian translation) and train a new adapter.

**Exercise 3**: Experiment with different target_modules and observe the impact on performance.

**Exercise 4**: Compare results with a larger model (GPT-2 medium) while maintaining the same LoRA parameters.

---

**End of Notebook** | Module 13.2 - LoRA Fine-tuning  
Data Visualization and Text Mining | Università Cattolica del Sacro Cuore